# RAG Benchmark: Tokens × Tempo

Este notebook lê os dados de execução de todos os RAGs diretamente do **LangSmith** e gera um gráfico comparativo de **consumo de tokens × latência** por query.

**Pré-requisito:** rodar cada RAG pelo menos uma vez com `LANGCHAIN_TRACING_V2=true` ativo antes de executar este notebook.

In [ ]:
import os
from datetime import datetime, timedelta, timezone

from langsmith import Client
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import numpy as np

In [ ]:
LANGSMITH_API_KEY = "XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX"

# Aumentar se os RAGs foram rodados há mais de 48h
LOOKBACK_HOURS = 48

RAG_PROJECTS = {
    "Context RAG":            "benchmark-context-rag",
    "Self RAG":               "benchmark-self-rag",
    "Hybrid RAG":             "benchmark-hybrid-rag",
    "Memory-Augmented RAG":   "benchmark-memory-augmented-rag",
    "Knowledge-Enhanced RAG": "benchmark-knowledge-enhanced-rag",
    "Graph RAG":              "benchmark-graph-rag",
}

client = Client(api_key=LANGSMITH_API_KEY)
print("LangSmith client inicializado.")

In [ ]:
def fetch_rag_metrics(project_name, lookback_hours=48):
    """Busca métricas de latência e tokens do LangSmith para um projeto RAG."""
    start_time = datetime.now(timezone.utc) - timedelta(hours=lookback_hours)
    latency_list, tokens_list = [], []

    # Busca runs raiz do tipo chain (LCEL chains, agents ou @traceable wrappers)
    root_runs = list(client.list_runs(
        project_name=project_name,
        run_type="chain",
        start_time=start_time,
        is_root=True,
    ))

    if not root_runs:
        print(f"  AVISO: Nenhum run encontrado no projeto '{project_name}'")
        print(f"  -> Rode o RAG com LANGCHAIN_TRACING_V2=true antes de usar este notebook.")
        return {"avg_tokens": None, "avg_latency_s": None, "n_runs": 0}

    for run in root_runs:
        # Latência: diferença entre end_time e start_time
        if run.end_time and run.start_time:
            latency_list.append((run.end_time - run.start_time).total_seconds())

        # Tokens: soma todos os LLM child runs dentro deste run raiz
        child_llm = list(client.list_runs(
            project_name=project_name,
            run_type="llm",
            parent_run_id=run.id,
        ))
        run_tokens = sum(
            (r.usage_metadata or {}).get("total_tokens", 0)
            for r in child_llm
        )
        if run_tokens > 0:
            tokens_list.append(run_tokens)

    return {
        "avg_tokens":    float(np.mean(tokens_list))  if tokens_list  else None,
        "avg_latency_s": float(np.mean(latency_list)) if latency_list else None,
        "n_runs":        len(root_runs),
    }


print("Buscando métricas do LangSmith...\n")
results = {}
for rag_name, project in RAG_PROJECTS.items():
    print(f"  {rag_name} ({project})...")
    results[rag_name] = fetch_rag_metrics(project, LOOKBACK_HOURS)
    m = results[rag_name]
    if m["avg_tokens"]:
        print(f"    runs={m['n_runs']}, avg_tokens={m['avg_tokens']:.0f}, avg_latência={m['avg_latency_s']:.2f}s")
    print()

In [ ]:
df = pd.DataFrame([
    {
        "RAG":             name,
        "Avg Tokens":      m["avg_tokens"],
        "Avg Latência (s)": m["avg_latency_s"],
        "N Queries":       m["n_runs"],
    }
    for name, m in results.items()
])

display(df)

In [ ]:
COLORS = ["#2c3e50", "#3498db", "#e74c3c", "#27ae60", "#f39c12", "#9b59b6"]

df_plot = df.dropna(subset=["Avg Tokens", "Avg Latência (s)"])

if df_plot.empty:
    print("Nenhum dado disponível para plotar. Rode os RAGs primeiro.")
else:
    fig, ax = plt.subplots(figsize=(11, 7))

    for i, row in df_plot.reset_index(drop=True).iterrows():
        color = COLORS[i % len(COLORS)]
        ax.scatter(
            row["Avg Latência (s)"],
            row["Avg Tokens"],
            s=220,
            color=color,
            zorder=5,
            edgecolors="white",
            linewidths=1.5,
        )
        ax.annotate(
            row["RAG"],
            xy=(row["Avg Latência (s)"], row["Avg Tokens"]),
            xytext=(10, 5),
            textcoords="offset points",
            fontsize=10,
            color=color,
            fontweight="bold",
        )

    # Linhas de mediana como referência visual
    if len(df_plot) > 1:
        ax.axvline(df_plot["Avg Latência (s)"].median(), color="gray", linestyle=":", alpha=0.5, label="Mediana latência")
        ax.axhline(df_plot["Avg Tokens"].median(),        color="gray", linestyle="--", alpha=0.5, label="Mediana tokens")
        ax.legend(fontsize=9, loc="upper left")

    ax.set_xlabel("Latência Média por Query (segundos)", fontsize=12)
    ax.set_ylabel("Tokens Médios por Query", fontsize=12)
    ax.set_title(
        "Benchmark RAG: Consumo de Tokens × Latência\n"
        "(canto inferior esquerdo = mais eficiente)",
        fontsize=14,
        fontweight="bold",
    )
    ax.grid(True, linestyle="--", alpha=0.4)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    plt.savefig("rag_benchmark_chart.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Gráfico salvo em: rag_benchmark_chart.png")